In [22]:
import pandas as pd

In [23]:
DATA_PATH = 'jobpostingdata.csv'
MODEL_PATH = 'model.pkl'
df = pd.read_csv(DATA_PATH)

In [24]:
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [25]:
def get_tag(tag:str):
    if tag[0] in ['V', 'N', 'R']:
        return tag[0].lower()
    elif tag[0] == 'J':
        return 'a'
    else:
        return 'n'
    
def lemmatization(tokens):
    lemmatizer = WordNetLemmatizer()
    lemmatized = []
    tagged = pos_tag(tokens)
    for word, tag in tagged:
        result = lemmatizer.lemmatize(word, get_tag(tag))
        lemmatized.append(result)
    return lemmatized

def text_preprocessor(sentence):
    eng_stopwords = stopwords.words('english')
    tokens = word_tokenize(sentence)
    cleaned = [
        token.lower() for token in tokens if token not in eng_stopwords
        and
        token.isalpha()
    ]
    lemmatized = lemmatization(cleaned)
    return lemmatized

In [26]:
def feature_extraction(sentence, existed_words):
    unique_words = set(text_preprocessor(sentence))
    feature = {}
    for word in existed_words:
        feature[word] = (word in unique_words)
    return feature

In [27]:
from nltk.classify.naivebayes import NaiveBayesClassifier
from nltk.classify import accuracy
from sklearn.model_selection import train_test_split

In [30]:
def train_classifier():
    x = df['text']
    y = df['fraudulent']
    corpus = ' '.join(x)
    existed_words = set(text_preprocessor(corpus))

    features = [
        (feature_extraction(sentence, existed_words), category)
        for sentence, category in zip(x, y)
    ]

    train_data, test_data = train_test_split(
        features, test_size=0.2, random_state=42
    )

    classifier = NaiveBayesClassifier.train(train_data)
    acc = accuracy(classifier, test_data)
    print(f"Accuracy: {acc*100:.2f}%")

    return classifier, existed_words

In [31]:
train_classifier()

Accuracy: 75.00%


(<nltk.classify.naivebayes.NaiveBayesClassifier at 0x20ab47bcf50>,
 {'ziele',
  'baby',
  'download',
  'cobble',
  'preferredclear',
  'opportunitiesstay',
  'scale',
  'comprise',
  'requiredexcellent',
  'close',
  'humanise',
  'α',
  'accountsoptimize',
  'orientedhotel',
  'ανεξάρτητης',
  'mathematics',
  'documentsmanage',
  'explore',
  'integrator',
  'africa',
  'synchronous',
  'andbanqueting',
  'magic',
  'hnc',
  'influencers',
  'ambulance',
  'scalable',
  'recipient',
  'refund',
  'join',
  'household',
  'teeth',
  'startup',
  'iptables',
  'voucherscycle',
  'stock',
  'influential',
  'systemmaintain',
  'notch',
  'managementa',
  'quarterly',
  'gartner',
  'coowe',
  'every',
  'pa',
  'essentially',
  'citigroup',
  'permanentlocation',
  'factor',
  'origin',
  'thread',
  'db',
  'neither',
  'chargeable',
  'clean',
  'carpet',
  'mitel',
  'cooperative',
  'updating',
  'lens',
  'net',
  'degreewill',
  'go',
  'browser',
  'bleed',
  'velcro',
  'filesm

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
def train_vectorizer():
    x = df['text']
    vectorizer = TfidfVectorizer(
        ngram_range=(1,2),
        stop_words='english'
    )
    vectors = vectorizer.fit_transform(x)
    return vectorizer, vectors

In [3]:
import os 
import pickle

In [4]:
def load_model():
    if os.path.exists(MODEL_PATH):
        with open(MODEL_PATH, 'rb') as f:
            classifier, existed_words, vectorizer, vectors = pickle.load(f)
        return classifier, existed_words, vectorizer, vectors
    else:
        classifier, existed_words = train_classifier()
        vectorizer, vectors = train_vectorizer()

        with open(MODEL_PATH, 'wb') as f:
            pickle.load((classifier, existed_words, vectorizer, vectors), f)
        return classifier, existed_words, vectorizer, vectors 

In [5]:
def print_menu(sentence, category):
    print(sentence)
    print(category)
    print("1. Write Sentence")
    print("2. Recommend")
    print("3. NER")

In [7]:
def write_sentence():
    sent = ''
    while len(sent.strip()) < 20 or len(sent.strip().split()) < 3:
        sent = input("Sentence: ")
    return sent

In [8]:
def classify_text(classifier, sentence, existed_words):
    feature = feature_extraction(sentence, existed_words)
    category = classifier.classify(feature)
    if category == 1:
        return 'TRUE'
    else:
        return 'FAKE'

In [9]:
def load_ner(nlp, sentence):
    doc = nlp(sentence)
    ents_dicts = {}

    for ent in doc.ents:
        if ent.label_ not in ents_dicts.keys():
            ents_dicts[ent.label_] = []
        ents_dicts[ent.label_].append(doc.text)
    return ents_dicts

In [10]:
def print_NER(ents_dicts):
    if ents_dicts:
        for key, value in ents_dicts.items():
            print(f"{key}: {value}")

In [1]:
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
def recommend_top_n(vectorizer: TfidfVectorizer, job_vectors, query, topn=5):
    vectorized_query = vectorizer.transform([query])
    similarity = cosine_similarity(job_vectors, vectorized_query).flatten()
    topidx = similarity.argsort()[::-1][:topn]

    for i, idx in enumerate(topidx, 1):
        print(f"{i}. {df['title'].iloc[idx]}")

NameError: name 'TfidfVectorizer' is not defined

In [3]:
import spacy

In [4]:
def menu():
    sent = ''
    cat = ''

    classifier = None
    existed_words = None
    vectorizer = None
    vectors = None
    nlp = spacy.load("en_core_web_sm")
    ents_dicts = {}

    while True:
        choice = print_menu(sent, cat).strip()

        if choice == '1':
            sent = write_sentence()
            if classifier is None or existed_words is None or vectorizer is None or vectors is None:
                classifier, existed_words, vectorizer, vectors = load_model()
        
        elif choice == '2':
            recommend_top_n(vectorizer, vectors, sent)

        elif choice == '3':
            ents_dicts = load_ner(nlp, sent)
            print_NER(ents_dicts)